In [1]:
import torch
import torch.nn as nn
import time
from pathlib import Path
import numpy as np
from model import TransformerLM, count_params
from model_utils import get_batch, cross_entropy, AdamW, save_checkpoint, load_checkpoint
from torch.profiler import profile, ProfilerActivity

project_dir = Path.cwd().parent
lm_dir = project_dir / "lm"
data_dir = project_dir / "data"
train_path = data_dir / "train.bin"
device = "cpu"

#model config
vocab_size = 10_000
context_length = 256
d_model = 512
num_layers = 4
num_heads = 16
num_kv_heads = 16
d_ff = 1344

#define seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [2]:
#importing train.bin
train_data = np.fromfile(train_path, dtype=np.uint16) 
print(f"Number of tokens: {len(train_data)}")
print(f"Min token ID: {train_data.min()}")
print(f"Max token ID: {train_data.max()}")

#converting to int64
train_data = train_data.astype(np.int64)

model = TransformerLM(
    vocab_size=vocab_size,
    context_length=context_length,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    num_kv_heads=num_kv_heads,
    d_ff=d_ff,
    tied=False,
).to(device)

print(f"Parameter count: {count_params(model)}")

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.0)

#sampling one batch
batch_size = 4
x, y = get_batch(train_data, batch_size, context_length, device)

print(f"Input shape: {x.shape}")
print(f"Target shape: {y.shape}")

Number of tokens: 540797225
Min token ID: 9
Max token ID: 9999
Parameter count: 22696448
Input shape: torch.Size([4, 256])
Target shape: torch.Size([4, 256])


In [6]:
max_steps = 300
log_every = 10
losses = []

model.train()

for step in range(1, max_steps + 1):
    optimizer.zero_grad()

    logits = model(x)
    loss = cross_entropy(logits, y)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}: {loss.item()}")

    loss.backward()

    # Check whether the gradients are finite
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=float("inf")
    )

    if not torch.isfinite(grad_norm):
        raise RuntimeError(f"Non-finite gradient norm at step {step}")

    optimizer.step()

    loss_value = loss.item()
    losses.append(loss_value)

    if step == 1 or step % log_every == 0:
        print(
            f"Step {step:3d} | "
            f"Loss: {loss_value:.4f} | "
            f"Gradient norm: {grad_norm.item():.4f}"
        )

    if loss_value < 0.1:
        print(f"\nTarget reached at step {step}: loss = {loss_value:.4f}")
        break
else:
    print(f"\nTarget not reached. Final loss: {losses[-1]:.4f}")


Step   1 | Loss: 9.2298 | Gradient norm: 1.5829
Step  10 | Loss: 2.6103 | Gradient norm: 1.3179
Step  20 | Loss: 0.4714 | Gradient norm: 0.3346

Target reached at step 27: loss = 0.0749


### part 2

In [7]:

# Check gradient norms over 50 training steps
grad_norms = []
losses_grad_check = []

model.train()

for step in range(1, 51):
    optimizer.zero_grad()

    logits = model(x)
    loss = cross_entropy(logits, y)
    loss.backward()

    # Measure the total gradient norm without clipping the gradients
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=float("inf")
    )

    grad_norm_value = grad_norm.item()

    if not np.isfinite(grad_norm_value):
        raise RuntimeError(
            f"Gradient norm is not finite at step {step}: {grad_norm_value}"
        )

    grad_norms.append(grad_norm_value)
    losses_grad_check.append(loss.item())

    optimizer.step()

    if step == 1 or step % 10 == 0:
        print(
            f"Step {step:2d} | "
            f"Loss: {loss.item():.4f} | "
            f"Gradient norm: {grad_norm_value:.4f}"
        )

print("\nGradient norm summary")
print(f"Minimum: {min(grad_norms):.4f}")
print(f"Maximum: {max(grad_norms):.4f}")
print(f"All finite: {np.isfinite(grad_norms).all()}")


Step  1 | Loss: 0.0564 | Gradient norm: 0.1007
Step 10 | Loss: 0.0092 | Gradient norm: 0.0191
Step 20 | Loss: 0.0033 | Gradient norm: 0.0075
Step 30 | Loss: 0.0019 | Gradient norm: 0.0047
Step 40 | Loss: 0.0013 | Gradient norm: 0.0035
Step 50 | Loss: 0.0010 | Gradient norm: 0.0028

Gradient norm summary
Minimum: 0.0028
Maximum: 0.1007
All finite: True


### part 3

In [9]:



# One complete training step
def train_step():
    model.train()
    optimizer.zero_grad()

    logits = model(x)
    loss = cross_entropy(logits, y)
    loss.backward()
    optimizer.step()

    return loss.item()

# Warm-up: these steps are not included in the profile
for _ in range(3):
    train_step()

# Profile five training steps
with profile(activities=[ProfilerActivity.CPU]) as prof:
    for _ in range(5):
        train_step()

# Show the operations that consumed the most CPU time
print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=20,
))

# Save a trace for visual inspection
prof.export_chrome_trace("trace.json")
print("\nTrace saved to trace.json")


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::bmm        41.05%        5.370s        41.06%        5.371s       9.678ms           555  
      autograd::engine::evaluate_function: BmmBackward0         5.88%     768.611ms        33.25%        4.350s      23.512ms           185  
                                           BmmBackward0         0.13%      17.569ms        27.37%        3.581s      19.358ms           185  
                              Optimizer.step#AdamW.step         2.72%     356.240ms        18.85%        2.466s     493.160ms             5  
      

### part 4

In [3]:

import json
import numpy as np
import torch

checkpoint_path = lm_dir / "resume_checkpoint.pt"
batch_path = lm_dir / "resume_batch.npz"
history_path = lm_dir / "resume_losses.json"

# Preserve the exact same batch across a runtime restart
np.savez(
    batch_path,
    x=x.detach().cpu().numpy(),
    y=y.detach().cpu().numpy(),
)

# Save the current model and optimizer state as step 0
save_checkpoint(model, optimizer, 0, checkpoint_path)
loss_history = []
with open(history_path, "w") as f:
    json.dump(loss_history, f)

print("Initial checkpoint saved.")
print("Starting training; checkpoints are saved every 5 steps.")

model.train()

for step in range(1, 16):
    optimizer.zero_grad()

    logits = model(x)
    loss = cross_entropy(logits, y)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if step % 5 == 0:
        # Save model, optimizer, and matching loss history together
        save_checkpoint(model, optimizer, step, checkpoint_path)

        with open(history_path, "w") as f:
            json.dump(loss_history, f)

        print(f"Checkpoint saved at step {step}; loss = {loss.item():.6f}")


Initial checkpoint saved.
Starting training; checkpoints are saved every 5 steps.
Checkpoint saved at step 5; loss = 5.526396
Checkpoint saved at step 10; loss = 2.610325
Checkpoint saved at step 15; loss = 1.243747


In [4]:

# Restore the exact batch used before the interruption
saved_batch = np.load(batch_path)
x = torch.tensor(saved_batch["x"], dtype=torch.long, device=device)
y = torch.tensor(saved_batch["y"], dtype=torch.long, device=device)

# Recreate the model with the same configuration
model = TransformerLM(
    vocab_size=10_000,
    context_length=256,
    d_model=512,
    num_layers=4,
    num_heads=16,
    num_kv_heads=16,
    d_ff=1344,
    tied=False,
).to(device)

optimizer = AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.0,
)

# Restore model weights, optimizer state, and saved step
start_step = load_checkpoint(checkpoint_path, model, optimizer)

with open(history_path, "r") as f:
    loss_history = json.load(f)

print("Resumed from step:", start_step)
print("Saved loss entries:", len(loss_history))
print("Last saved loss:", loss_history[-1] if loss_history else "None")


Resumed from step: 15
Saved loss entries: 15
Last saved loss: 1.2437468767166138


In [ ]:

model.train()

for step in range(start_step + 1, 31):
    optimizer.zero_grad()

    logits = model(x)
    loss = cross_entropy(logits, y)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if step % 5 == 0:
        save_checkpoint(model, optimizer, step, checkpoint_path)

        with open(history_path, "w") as f:
            json.dump(loss_history, f)

        print(f"Step {step} | Loss: {loss.item():.6f}")

print("Training resumed and completed.")


Step 20 | Loss: 0.471394
Step 25 | Loss: 0.133125
